# Manuel de reproduction & guide technique PE_V2X_Reliability (Approche A)

Ce manuel vise la reproduction par un évaluateur et la lecture du code : étapes de reproduction indépendantes du chemin, contrat de sortie run_id, limites de modification des paramètres, et guide fichier par fichier (scripts de haut niveau + modules complets).

**Date:** 2026-03-01


## 1 Exécution de reproduction (terminal intégré VS Code ; indépendant du chemin)

Il est recommandé d’ouvrir la racine du dépôt dans VS Code (notée `<ROOT>`), puis *Terminal → New Terminal*. Vous pouvez choisir **Command Prompt / PowerShell** ; deux écritures courantes sont proposées ci-dessous (au choix).

### 1.1 Créer et activer un environnement virtuel (Windows)

**Variante A : Command Prompt (recommandée)**

```bat
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\activate.bat
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

**Variante B : PowerShell (optionnelle)**

```powershell
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\Activate.ps1
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

> Remarque : si PowerShell bloque l’exécution des scripts, exécuter (une fois)  
> `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`

### 1.2 Smoke test (vérification rapide “sans erreur”)

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Smoke --scenarios Ref --rets 0 --n_seeds 1 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
```

Les sorties attendues se trouvent dans :
`<ROOT>\05_results_A\runs\A_Smoke\` (contrat en Section 2).

### 1.3 Small-seeds (sanity check avant les longs runs)

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Small --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 3 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
python run_pipeline_A.py --run_id A_Small_Cong --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 3 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 1
```

### 1.4 Run final (S20) — identique aux figures du rapport

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Final_NoCong_S20 --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 20 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
python run_pipeline_A.py --run_id A_Final_Cong_S20   --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 20 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 1
```


## 2 Contrat de sortie run_id (check-list d’acceptation)

Dossier de sortie :
`<ROOT>\05_results_A\runs\<run_id>\`

Doit contenir :
- `raw/` : CSV paquet (potentiellement volumineux)
- `tables/` : CSV agrégés (préférés en relecture)
- `figures/` : figures de soutien F1/F2/F3
- `run_commands.txt` : instantané de commande (preuve de reproductibilité)

Recommandations :
1) `tables/` et `run_commands.txt` doivent exister ;
2) UrbMask doit produire `position_heterogeneity__*.csv` ; Tunnel doit produire `tunnel_segments__*.csv` ;
3) Si stockage limité, `raw/` peut être supprimé, mais conserver `tables/` et `run_commands.txt` est recommandé.


## 3 Limites de modification des paramètres (un paramètre à la fois + validation small-seeds)

- **Couche sûre (recommandée)** : ne modifier que les paramètres CLI (`rets` / `msg_rate_hz` / `deadline_ms` / `n_seeds` / `enable_congestion`).
- **Risque moyen** : modifier `scenario_*.py` ou les modules de propagation (`prop_city.py` / `prop_tunnel.py`).
- **Haut risque** : modifier `mac_congestion.py` (change la structure Cong-only) ; après modification, revalider prioritairement Fig03/Fig04.

Pour reproduire le tracé MATLAB de Fig01–Fig07, on peut re-tracer depuis le cache `.mat` sous `assets/matlab_cache_raw/`. Pour la relecture, les aperçus PNG et le PDF final suffisent.


## 4 Guide code fichier par fichier (scripts + modules)

Le contenu ci-dessous suit l’ossature « objectif — structure clé — E/S — points d’entrée de paramétrage », afin de faciliter le repérage dès la première lecture.


### 4.1 Scripts de haut niveau (03_sim_A/py)

### analyze_metrics_A.py

**Objectif** : agrégation : raw → tables (système à trois tables) et génération de CSV adaptés à la relecture.
- Taille : ~ **335 lignes**
- Paramètres CLI (extrait) : `--band_max_m`, `--band_min_m`, `--dist_bin_m`, `--mid_bin_m`, `--min_success_per_bin`, `--min_total_per_bin`, `--retrans`, `--run_id`, `--scenario`, `--u_bin_w`, `--u_max`, `--u_min`

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés (repérage rapide)** :
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `_quantiles_ms(x)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `main()`

**Fichiers/produits possibles (extrait)** :
- `position_heterogeneity__UrbMask__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`
- `results_packets__{scenario}__ret{ret}.csv`
- `results_packets__{scenario}__ret{ret}__seed*.csv`
- `summary_metrics__{args.scenario}__ret{args.retrans}__{tag}.csv`
- `tunnel_segments__Tunnel__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`

**Points d’entrée (un paramètre à la fois ; validation small-seeds)** :
- Préférer preset/CLI ; si des constantes internes sont modifiées, les consigner dans le rapport et dans les instantanés de commandes.

### generate_trajectories_A.py

**Objectif** : génération d’entrée : CSV de trajectoires (RefPlus : géométrie + IDM + feux + cross/turn).
- Taille : ~ **649 lignes**
- Paramètres CLI (extrait) : `--cross_enable`, `--cross_half_length_m`, `--dt_s`, `--duration_s`, `--flow_cross_vph`, `--flow_main_vph`, `--i1_x`, `--i2_x`, `--idm_T_s`, `--idm_a_mps2`, `--idm_b_mps2`, `--idm_delta`…

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.road_geometry.RefPlusGeometry`, `modules.traffic_signals.SignalPlan`, `modules.traffic_idm.IDMParams`, `modules.traffic_idm.idm_accel`

**Structures clés** :
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `_ensure_time_key(df)`
  - `_spawn_ok(vlist, min_spawn_gap_m)`
  - `_road_tag_main(direction, lane_id)`
  - `_road_tag_cross(which, direction, lane_id)`
  - `_road_tag_turn(which, turn_kind, lane_id)`
  - `_lane_key(kind, which, direction, lane_id)`
  - `_init_lane_state(geom, n_lanes_per_dir, cross_enable)`
  - `_simulate_refplus_idm(geom, plan_i1, plan_i2, flow_main_vph, flow_cross_vph, p_turn_i1, p_turn_i2, p_right, p_left, veh_length_m…)`
  - `main()`

**Fichiers/produits** :
- `traj__Ref.csv`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### generate_tunnel_config_A.py

**Objectif** : génération d’entrée : CSV de configuration Tunnel (intervalle, transitions, intensité, paramètres de queue).
- Taille : ~ **77 lignes**
- Paramètres CLI (extrait) : `--b_floor`, `--b_peak`, `--bell_gamma`, `--delay_exp_scale_ms`, `--delay_extra_ms`, `--run_id`, `--scenario`, `--severity`, `--transition_m`, `--x0_m`, `--x1_m`

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.prop_tunnel.TunnelConfig`

**Structures clés** :
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Fichiers/produits** :
- `tunnel_config__{args.scenario}.csv`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### generate_urbmask_buildings_A.py

**Objectif** : génération d’entrée : CSV bâtiments UrbMask (rectangles + hauteurs).
- Taille : ~ **83 lignes**
- Paramètres CLI (extrait) : `--max_h_m`, `--max_height_m`, `--max_w_m`, `--min_h_m`, `--min_height_m`, `--min_w_m`, `--n_blocks`, `--road_length_m`, `--run_id`, `--scenario`, `--seed`, `--x_margin_m`…

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.buildings_3d.generate_buildings`, `modules.buildings_3d.save_buildings_csv`

**Structures clés** :
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Fichiers/produits** :
- `buildings__{args.scenario}__seed{args.seed}.csv`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### paths_A.py

**Objectif** : conventions de chemins : gestion centralisée des répertoires et règles de nommage.
- Taille : ~ **156 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `datetime.datetime`, `json`

**Structures clés** :
- Classes :
  - `BasePathsA`
  - `RunPathsA`
- Fonctions :
  - `_detect_project_root(from_file)`
  - `get_base_paths_a()`
  - `make_run_id(prefix)`
  - `ensure_base_dirs_a()`
  - `ensure_run_dirs_a(run_id, save_as_latest, meta)`
  - `load_latest_run_id()`

**Fichiers/produits** :
- `LATEST_RUN.json`
- `run_manifest.json`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### plot_figures_A.py

**Objectif** : figures de soutien : produire F1/F2/F3 depuis les tables.
- Taille : ~ **264 lignes**
- Paramètres CLI (extrait) : `--cdf_max_dist_m`, `--curve_style`, `--min_bin_count`, `--retrans`, `--run_id`, `--scenario`, `--smooth_overlay_raw`, `--smooth_window_m`, `--x_max_m`

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `matplotlib.pyplot`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés** :
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `ecdf(x)`
  - `_pick_latest_summary_file(tables_dir, scenario, ret)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `_smooth_by_distance(x, y, window_m)`
  - `main()`

**Fichiers/produits** :
- `F1_PDR_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `F2_Delay_CDF__{scenario}__ret{ret}__{tag_f2}.png`
- `F3_Delay_p95_p99_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `results_packets__{scenario}__ret{ret}.csv`
- `summary_metrics__{scenario}__ret{ret}__*.csv`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### progress_util.py

**Objectif** : outils de progression : barre de progression et temps écoulé.
- Taille : ~ **43 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `typing.Iterable`, `typing.Iterator`, `typing.Optional`, `typing.TypeVar`, `sys`

**Structures clés** :
- Fonctions :
  - `progress(it, total, desc)`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### run_logging.py

**Objectif** : audit : écrire run_commands / manifest, etc.
- Taille : ~ **97 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `json`, `os`, `shutil`, `sys`, `datetime.datetime`, `pathlib.Path`, `typing.Any`, `typing.Dict`, `typing.Optional`

**Structures clés** :
- Fonctions :
  - `_now()`
  - `log_command(run_id, run_results_dir, extra)`
  - `update_manifest(manifest_path, patch)`
  - `snapshot_file(src, run_results_dir, category, rename_to)`
  - `_quote(s)`

**Fichiers/produits** :
- `run_commands.txt`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### run_pipeline_A.py

**Objectif** : orchestrateur : contrat run_id + génération d’entrées → simulation → agrégation → figures → audit.
- Taille : ~ **533 lignes**
- Paramètres CLI (extrait) : `--buildings_seed`, `--cross_enable`, `--cross_half_length_m`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us`…

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `subprocess`, `sys`, `pathlib.Path`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés** :
- Fonctions :
  - `_run(cmd, cwd)`
  - `main()`

**Points d’entrée** :
- Recommandation : modifier en priorité via le CLI de `run_pipeline_A.py` (`--scenarios/--rets/--n_seeds/--msg_rate_hz/--deadline_ms/--enable_congestion`, etc.).

### sim_v2x_A.py

**Objectif** : cœur de simulation : échantillons paquet (distance/succès/délai) + champs de preuve masquage/tunnel/congestion.
- Taille : ~ **642 lignes**
- Paramètres CLI (extrait) : `--attempt_spacing_ms`, `--buildings_path`, `--buildings_seed`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us`…

**Dépendances principales (extrait)** :
`__future__.annotations`, `argparse`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `typing.Optional`, `numpy`, `pandas`, `progress_util.progress`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `paths_A.ensure_base_dirs_a`, `paths_A.get_base_paths_a`, `modules.mac_congestion.CongestionParams`, `modules.mac_congestion.compute_ncs_from_distances`, `modules.mac_congestion.compute_airtime_s`, `modules.mac_congestion.compute_cbr`, `modules.mac_congestion.p_collision_from_ncs`, `modules.mac_congestion.congestion_extra_delay_ms` …

**Structures clés** :
- Classes :
  - `Rect`
- Fonctions :
  - `clamp01(x)`
  - `load_traj(traj_path)`
  - `load_buildings(buildings_path)`
  - `_legacy_dirs()`
  - `_pick_run_id(arg_run_id)`
  - `parse_tx_ids(s, all_ids)`
  - `_tag_is_cross(tag, prefixes)`
  - `compute_delay_ms(distance_m, attempt_idx, attempt_spacing_ms, rng, impairment_b, extra_ms, exp_scale_ms)`
  - `simulate_one_seed(scenario, retrans, seed, msg_rate_hz, tx_ids_fixed, tx_mode, tx_k, tx_k_cross, tx_cross_prefixes, traj…)`
  - `main()`

**Fichiers/produits** :
- `buildings__UrbMask__seed{seed_use}.csv`
- `results_packets__{args.scenario}__ret{args.retrans}__{seed_tag}.csv`
- `traj__Ref.csv`
- `traj__{args.scenario}.csv`
- `tunnel_config__Tunnel.csv`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.


### 4.2 Modules (03_sim_A/py/modules)

### road_geometry.py

**Objectif** : géométrie routière : ligne centrale/voies/carrefours/branches, construction et requêtes.
- Taille : ~ **194 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `typing.Union`, `numpy`

**Structures clés** :
- Classes :
  - `RefPlusGeometry`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### mac_congestion.py

**Objectif** : proxys de congestion : n_cs/CBR/p_col et cong_delay_ms (incluant structure de queue).
- Taille : ~ **166 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `typing.Optional`, `numpy`

**Structures clés** :
- Classes :
  - `CongestionParams`
- Fonctions :
  - `compute_airtime_s(pkt_bytes, phy_rate_mbps, mac_efficiency, phy_overhead_us)`
  - `compute_cbr(n_cs, msg_rate_hz, airtime_s, cbr_cap)`
  - `p_collision_from_ncs(n_cs, alpha_col, cbr, gamma_cbr_col)`
  - `congestion_extra_delay_ms(rng, n_cs, beta_delay_ms, exp_scale_ms, cbr, gamma_cbr_delay)`
  - `compute_ncs_from_distances(dist_all, tx_index, r_cs_m, active_mask, speed_all, min_speed_mps)`

**Points d’entrée** :
- Surveiller la chaîne `n_cs→CBR→p_col` et la moyenne/queue de `cong_delay_ms` ; revalider Fig03/Fig04 après toute modification.

### buildings_3d.py

**Objectif** : géométrie bâtiments : génération de blocs et requêtes (support de d_min).
- Taille : ~ **163 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `numpy`, `pandas`

**Structures clés** :
- Classes :
  - `Rect2D`
  - `BuildingBlock` (hérite de Rect2D)
- Fonctions :
  - `_pick_zone_by_x(x_mid, road_length_m)`
  - `generate_buildings()`
  - `buildings_to_dataframe(buildings)`
  - `save_buildings_csv(buildings, path)`
  - `load_buildings_csv(path)`
  - `as_rects(buildings)`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### prop_city.py

**Objectif** : dégradation urbaine : d_min→b→mélange LOS/NLOS, terme de réflexion équivalent optionnel.
- Taille : ~ **140 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `typing.Iterable`, `typing.Protocol`, `typing.Tuple`, `numpy`

**Structures clés** :
- Classes :
  - `RectLike` (Protocol)
- Fonctions :
  - `clamp01(x)`
  - `segment_intersects_rect(ax, ay, bx, by, rect)`
  - `segment_to_rect_min_distance(ax, ay, bx, by, rect)`
  - `blockage_strength_with_dmin(ax, ay, bx, by, buildings, transition_m)`
  - `p_success_los(distance_m)`
  - `p_success_nlos(distance_m)`
  - `refl_gain_db(d_min_m, b, gmax_db, d0_m)`

**Points d’entrée** :
- Surveiller l’échelle `d_min→b` (p. ex. `T/urb_transition_m`), les courbes LOS/NLOS, et les coefficients du terme de réflexion.

### prop_tunnel.py

**Objectif** : dégradation tunnel : position tunnel_u ; bell+fade ; injection de queue.
- Taille : ~ **111 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Tuple`, `numpy`, `pandas`

**Structures clés** :
- Classes :
  - `TunnelConfig`
- Fonctions :
  - `clamp01(x)`
  - `tunnel_impairment_b(tx_x, rx_x, cfg)`

**Points d’entrée** :
- Surveiller intervalle, longueur de transition, paramètres d’intensité (`b_floor/b_peak/γ`) et l’échelle de queue.

### scenario_refplus.py

**Objectif** : preset de scénario : geler les paramètres par défaut (géométrie/trafic/propagation/tunnel/congestion).
- Taille : ~ **65 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Structures clés** :
- Classes :
  - `RefPlusGeometryConfig`
  - `TrafficIDMConfig`
  - `TrafficSignalConfig`
  - `RefPlusScenarioConfig`

**Points d’entrée** :
- Modifier les défauts dans `scenario_*.py` ; revalider les tendances Fig03/04/05/06/07.

### traffic_signals.py

**Objectif** : dynamique du trafic : plans de feux (phase/offset), support des files et libérations.
- Taille : ~ **54 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`

**Structures clés** :
- Classes :
  - `SignalPlan`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### scenario_urbmaskplus.py

**Objectif** : preset de scénario : geler les paramètres par défaut (géométrie/trafic/propagation/tunnel/congestion).
- Taille : ~ **43 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Structures clés** :
- Classes :
  - `UrbMaskBuildingsConfig`
  - `UrbMaskPropagationConfig`
  - `UrbMaskScenarioConfig`

**Points d’entrée** :
- Modifier dans `scenario_*.py` ; revalider Fig03/04/05/06/07.

### traffic_idm.py

**Objectif** : dynamique du trafic : suivi IDM (support des files/pelotons).
- Taille : ~ **38 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `numpy`

**Structures clés** :
- Classes :
  - `IDMParams`
- Fonctions :
  - `idm_accel(v, dv, gap, p)`

**Points d’entrée** :
- Préférer preset/CLI ; consigner toute modification interne.

### scenario_tunnelplus.py

**Objectif** : preset de scénario : geler les paramètres par défaut (géométrie/trafic/propagation/tunnel/congestion).
- Taille : ~ **17 lignes**

**Dépendances principales (extrait)** :
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`, `prop_tunnel.TunnelConfig`

**Structures clés** :
- Classes :
  - `TunnelScenarioConfig`

**Points d’entrée** :
- Modifier dans `scenario_*.py` ; revalider Fig03/04/05/06/07.
